In [1]:
import pickle
import numpy as np
import tensorflow as tf
import keras
from cnn_base import build_t1, build_t1_int, prepare_input, export_integer_model, integer_inference
import os


TRAIN_PICKLE_PATH = "./dataset/sat6_train.pkl"
TEST_PICKLE_PATH = "./dataset/sat6_test.pkl"
DATA_DIR = "./dataset/sat-6-full.mat"
BATCH_SIZE = 512
EPOCHS = 30
LEARNING_RATE = 3e-4
 
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

def trace_layers(config):
    config['Model']['TraceOutput'] = True
    for layer in config['LayerName']:
        config['LayerName'][layer]['Trace'] = True


def print_dict(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + str(key), end='')
        if isinstance(value, dict):
            print()
            print_dict(value, indent + 1)
        else:
            print(':' + ' ' * (20 - len(key) - 2 * indent) + str(value))
 
def load_pickle_data(path):
    print(f"Lendo {path} ...")
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"], data["y"]

2026-10-29 22:26:35.808894: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-29 22:26:35.820151: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-29 22:26:35.947922: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-29 22:26:35.948937: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-29 22:26:36.410257: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:3

In [2]:
X_train_all, y_train_all = load_pickle_data(TRAIN_PICKLE_PATH)
X_test, y_test = load_pickle_data(TEST_PICKLE_PATH)

# treino inteiro: entrada uint8 bruta 0..255, sem normalização

val_idx = int(0.75 * len(X_train_all))
X_train, y_train = X_train_all[:val_idx], y_train_all[:val_idx]
X_val, y_val = X_train_all[val_idx:], y_train_all[val_idx:]

n_classes = 6
input_shape = X_train.shape[1:]
train_size = len(X_train)
input_shape, train_size

Lendo ./dataset/sat6_train.pkl ...
Lendo ./dataset/sat6_test.pkl ...


((32, 32, 4), 243000)

# Train T1 base model

In [3]:
# Finn base model not quantized
model = build_t1()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy")
    ],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    #keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
]

model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks,  batch_size=BATCH_SIZE)


os.makedirs('models', exist_ok=True)
os.makedirs('models/t1', exist_ok=True)
base_path = './models/t1/'
model.save(base_path + '/model.keras')
model.save(base_path + '/model.h5')
model.save_weights(base_path + '/weights.h5')

with open(base_path + '/model.json', 'w') as f:
    f.write(model.to_json())

Epoch 1/30
475/475 [==============================] - 2s 5ms/step - loss: 1.8589 - accuracy: 0.8172 - val_loss: 0.3601 - val_accuracy: 0.9174
Epoch 2/30
475/475 [==============================] - 2s 4ms/step - loss: 0.2907 - accuracy: 0.9284 - val_loss: 0.2182 - val_accuracy: 0.9424
Epoch 3/30
475/475 [==============================] - 2s 4ms/step - loss: 0.2035 - accuracy: 0.9441 - val_loss: 0.1924 - val_accuracy: 0.9440
Epoch 4/30
475/475 [==============================] - 2s 4ms/step - loss: 0.1688 - accuracy: 0.9496 - val_loss: 0.1567 - val_accuracy: 0.9522
Epoch 5/30
475/475 [==============================] - 2s 4ms/step - loss: 0.1430 - accuracy: 0.9545 - val_loss: 0.1420 - val_accuracy: 0.9550
Epoch 6/30
475/475 [==============================] - 2s 4ms/step - loss: 0.1291 - accuracy: 0.9571 - val_loss: 0.1245 - val_accuracy: 0.9591
Epoch 7/30
475/475 [==============================] - 2s 4ms/step - loss: 0.1144 - accuracy: 0.9598 - val_loss: 0.1115 - val_accuracy: 0.9610
Epoch 

/home/fauri/Documentos/hls4ml/.venv_hls4ml_t/lib/python3.10/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [4]:
# Finn base model not quantized with max pooling and batch normalization
model = build_t1(max_pool=True, batch_norm=True)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy")
    ],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    #keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
]

model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks,  batch_size=BATCH_SIZE)


os.makedirs('models', exist_ok=True)
os.makedirs('models/t1_maxpool_bn', exist_ok=True)
base_path = './models/t1_maxpool_bn/'
model.save(base_path + '/model.keras')
model.save(base_path + '/model.h5')
model.save_weights(base_path + '/weights.h5')

with open(base_path + '/model.json', 'w') as f:
    f.write(model.to_json())

Epoch 1/30
475/475 [==============================] - 2s 5ms/step - loss: 0.5187 - accuracy: 0.8350 - val_loss: 0.2431 - val_accuracy: 0.9255
Epoch 2/30
475/475 [==============================] - 2s 4ms/step - loss: 0.1349 - accuracy: 0.9560 - val_loss: 0.1139 - val_accuracy: 0.9616
Epoch 3/30
475/475 [==============================] - 2s 4ms/step - loss: 0.0972 - accuracy: 0.9679 - val_loss: 0.0833 - val_accuracy: 0.9721
Epoch 4/30
475/475 [==============================] - 2s 4ms/step - loss: 0.0788 - accuracy: 0.9739 - val_loss: 0.0794 - val_accuracy: 0.9726
Epoch 5/30
475/475 [==============================] - 2s 4ms/step - loss: 0.0679 - accuracy: 0.9774 - val_loss: 0.0709 - val_accuracy: 0.9750
Epoch 6/30
475/475 [==============================] - 2s 4ms/step - loss: 0.0610 - accuracy: 0.9797 - val_loss: 0.0721 - val_accuracy: 0.9758
Epoch 7/30
475/475 [==============================] - 2s 4ms/step - loss: 0.0554 - accuracy: 0.9817 - val_loss: 0.0653 - val_accuracy: 0.9779
Epoch 

# Train T1 Integer

Pesos int8 (escala 2^k por canal), ativações uint8, acumulador int32. `bits=4` vira int4/uint4.

In [5]:
bits = [8, 4]

for bit in bits:
    model = build_t1_int(
        bits=bit,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="accuracy")
        ],
    )
    callbacks = [
        keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True, verbose=1),
        #keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ]

    model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks, batch_size=BATCH_SIZE)


    os.makedirs('models', exist_ok=True)
    model_name = f't1_{bit}'
    os.makedirs(f'models/{model_name}', exist_ok=True)
    base_path = f'./models/{model_name}/'
    model.save(base_path + '/model.keras')
    model.save(base_path + '/model.h5')
    model.save_weights(base_path + '/weights.h5')

    with open(base_path + '/model.json', 'w') as f:
        f.write(model.to_json())

Epoch 1/30
475/475 [==============================] - 8s 17ms/step - loss: 3.5351 - accuracy: 0.7897 - val_loss: 0.8015 - val_accuracy: 0.8946
Epoch 2/30
475/475 [==============================] - 8s 16ms/step - loss: 0.6572 - accuracy: 0.9051 - val_loss: 0.4833 - val_accuracy: 0.9167
Epoch 3/30
475/475 [==============================] - 8s 16ms/step - loss: 0.4370 - accuracy: 0.9201 - val_loss: 0.4033 - val_accuracy: 0.9190
Epoch 4/30
475/475 [==============================] - 8s 16ms/step - loss: 0.3189 - accuracy: 0.9295 - val_loss: 0.2534 - val_accuracy: 0.9389
Epoch 5/30
475/475 [==============================] - 8s 16ms/step - loss: 0.2512 - accuracy: 0.9354 - val_loss: 0.2375 - val_accuracy: 0.9347
Epoch 6/30
475/475 [==============================] - 8s 16ms/step - loss: 0.2054 - accuracy: 0.9418 - val_loss: 0.1880 - val_accuracy: 0.9456
Epoch 7/30
475/475 [==============================] - 8s 16ms/step - loss: 0.1739 - accuracy: 0.9480 - val_loss: 0.1557 - val_accuracy: 0.9514

# T1 Integer with pooling and batch norm (BN dobrada nos pesos inteiros)

In [6]:
bits = [8, 4]

for bit in bits:
    model = build_t1_int(
        bits=bit,
        max_pool=True,
        batch_norm=True,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="accuracy")
        ],
    )
    callbacks = [
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
        #keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ]

    model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks, batch_size=BATCH_SIZE)


    os.makedirs('models', exist_ok=True)
    model_name = f't1_{bit}_maxpool_bn'
    os.makedirs(f'models/{model_name}', exist_ok=True)
    base_path = f'./models/{model_name}/'
    model.save(base_path + '/model.keras')
    model.save(base_path + '/model.h5')
    model.save_weights(base_path + '/weights.h5')

    with open(base_path + '/model.json', 'w') as f:
        f.write(model.to_json())

Epoch 1/30
475/475 [==============================] - 12s 23ms/step - loss: 5.0122 - accuracy: 0.7756 - val_loss: 1.4090 - val_accuracy: 0.8710
Epoch 2/30
475/475 [==============================] - 11s 23ms/step - loss: 0.9617 - accuracy: 0.8955 - val_loss: 0.7404 - val_accuracy: 0.9138
Epoch 3/30
475/475 [==============================] - 11s 23ms/step - loss: 0.5951 - accuracy: 0.9213 - val_loss: 0.5881 - val_accuracy: 0.9266
Epoch 4/30
475/475 [==============================] - 11s 23ms/step - loss: 0.4536 - accuracy: 0.9333 - val_loss: 0.3995 - val_accuracy: 0.9370
Epoch 5/30
475/475 [==============================] - 11s 23ms/step - loss: 0.3770 - accuracy: 0.9404 - val_loss: 0.3383 - val_accuracy: 0.9481
Epoch 6/30
475/475 [==============================] - 11s 23ms/step - loss: 0.3102 - accuracy: 0.9464 - val_loss: 0.2998 - val_accuracy: 0.9447
Epoch 7/30
475/475 [==============================] - 11s 23ms/step - loss: 0.2643 - accuracy: 0.9504 - val_loss: 0.2458 - val_accuracy:

# Conferir que o modelo é inteiro

Refaz a inferência só com aritmética inteira (numpy) a partir dos inteiros exportados e compara com o Keras.

In [7]:
from qkeras.utils import load_qmodel

for model_name in ['t1_8', 't1_4', 't1_8_maxpool_bn', 't1_4_maxpool_bn']:
    qmodel = load_qmodel(f'models/{model_name}/model.h5')
    X = prepare_input(X_test[:2000], qmodel)

    ops = export_integer_model(qmodel)
    logits_int = integer_inference(ops, X)
    logits_keras = keras.Model(qmodel.input, qmodel.get_layer('output_dense').output).predict(X, verbose=0)

    print(f'=== {model_name}')
    for op in ops:
        if 'w' in op:
            print(f"  {op['name']:13s} pesos int [{op['w'].min()}, {op['w'].max()}]  shift 2^k, k em [{op['k'].min()}, {op['k'].max()}]")
    print(f'  max|logits int - keras| = {np.abs(logits_int - logits_keras).max():.2e}'
          f'  argmax igual: {(logits_int.argmax(1) == logits_keras.argmax(1)).mean():.4f}'
          f'  acc inteira: {(logits_int.argmax(1) == y_test[:2000].argmax(1)).mean():.4f}')

=== t1_8
  conv1         pesos int [-127, 127]  shift 2^k, k em [-10, -8]
  conv2         pesos int [-127, 127]  shift 2^k, k em [-8, -7]
  output_dense  pesos int [-127, 121]  shift 2^k, k em [-9, -9]
  max|logits int - keras| = 0.00e+00  argmax igual: 1.0000  acc inteira: 0.9785
=== t1_4
  conv1         pesos int [-7, 7]  shift 2^k, k em [-9, -6]
  conv2         pesos int [-7, 7]  shift 2^k, k em [-4, -3]
  output_dense  pesos int [-7, 7]  shift 2^k, k em [-5, -4]
  max|logits int - keras| = 0.00e+00  argmax igual: 1.0000  acc inteira: 0.9745
=== t1_8_maxpool_bn
  conv1         pesos int [-127, 127]  shift 2^k, k em [-11, -8]
  conv2         pesos int [-127, 127]  shift 2^k, k em [-8, -7]
  output_dense  pesos int [-127, 127]  shift 2^k, k em [-9, -8]
  max|logits int - keras| = 0.00e+00  argmax igual: 1.0000  acc inteira: 0.9820
=== t1_4_maxpool_bn
  conv1         pesos int [-7, 7]  shift 2^k, k em [-10, -7]
  conv2         pesos int [-7, 7]  shift 2^k, k em [-4, -3]
  output_dense 